Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_lstm_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 6)
Las dimensiones de testX son:  (10529, 12, 6)
Las dimensiones de valX son:  (5186, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [13]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [14]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 22s - 383ms/step - ia: 0.3316 - loss: 1.1832 - mae: 0.8548 - rmse: 1.0853 - smape: 1.3756 - val_ia: 0.1883 - val_loss: 0.6761 - val_mae: 0.6892 - val_rmse: 0.8183 - val_smape: 1.0783

Epoch 2/128                                           

58/58 - 1s - 23ms/step - ia: 0.2837 - loss: 1.0327 - mae: 0.8110 - rmse: 1.0145 - smape: 1.4533 - val_ia: 0.2656 - val_loss: 0.7427 - val_mae: 0.7177 - val_rmse: 0.8554 - val_smape: 1.2092

Epoch 3/128                                           

58/58 - 2s - 30ms/step - ia: 0.2469 - loss: 0.9556 - mae: 0.7895 - rmse: 0.9749 - smape: 1.5137 - val_ia: 0.2959 - val_loss: 0.8148 - val_mae: 0.7545 - val_rmse: 0.8943 - val_smape: 1.3827

Epoch 4/128                                           

58/58 - 4s - 70ms/step - ia: 0.2188 - loss: 0.9392 - mae: 0.7876 - rmse: 0.9673 - smape: 1.5514 - val_ia: 0.3055 - val_loss: 0.8712 - val_mae: 0.7833 - val_rmse: 0.9239 - val_smape: 1.5503

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                     

461/461 - 40s - 86ms/step - ia: 0.7564 - loss: 0.1773 - mae: 0.3135 - rmse: 0.3964 - smape: 0.6366 - val_ia: 0.3494 - val_loss: 0.1930 - val_mae: 0.3480 - val_rmse: 0.3776 - val_smape: 0.6816

Epoch 2/128                                                                     

461/461 - 18s - 39ms/step - ia: 0.8511 - loss: 0.0728 - mae: 0.2037 - rmse: 0.2625 - smape: 0.4857 - val_ia: 0.3727 - val_loss: 0.1288 - val_mae: 0.2866 - val_rmse: 0.3116 - val_smape: 0.5585

Epoch 3/128                                                                     

461/461 - 13s - 29ms/step - ia: 0.8700 - loss: 0.0565 - mae: 0.1784 - rmse: 0.2311 - smape: 0.4525 - val_ia: 0.4125 - val_loss: 0.0979 - val_mae: 0.2463 - val_rmse: 0.2705 - val_smape: 0.5219

Epoch 4/128                                                                     

461/461 - 11s - 24ms/step - ia: 0.8775 - loss: 0.0495 - mae: 0.1673 - rmse: 0.2167 - smape: 0

...

Epoch 1/128                                                                       

461/461 - 12s - 26ms/step - ia: 0.2526 - loss: 0.9445 - mae: 0.7949 - rmse: 0.9606 - smape: 1.5007 - val_ia: 0.1849 - val_loss: 0.9495 - val_mae: 0.8197 - val_rmse: 0.8500 - val_smape: 1.8362

Epoch 2/128                                                                       

461/461 - 6s - 13ms/step - ia: 0.2543 - loss: 0.8612 - mae: 0.7622 - rmse: 0.9168 - smape: 1.5155 - val_ia: 0.1913 - val_loss: 0.8821 - val_mae: 0.7833 - val_rmse: 0.8154 - val_smape: 1.6178

Epoch 3/128                                                                       

461/461 - 10s - 22ms/step - ia: 0.2766 - loss: 0.7924 - mae: 0.7311 - rmse: 0.8792 - smape: 1.4765 - val_ia: 0.1873 - val_loss: 0.8048 - val_mae: 0.7422 - val_rmse: 0.7737 - val_smape: 1.4310

Epoch 4/128                                                                       

461/461 - 6s - 13ms/step - ia: 0.3531 - loss: 0.6804 - mae: 0.6773 - rmse: 0.8137 - sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

922/922 - 11s - 12ms/step - ia: 0.2450 - loss: 0.8740 - mae: 0.7654 - rmse: 0.9100 - smape: 1.6300 - val_ia: 0.1425 - val_loss: 0.8056 - val_mae: 0.7530 - val_rmse: 0.7663 - val_smape: 1.4286

Epoch 2/128                                                                          

922/922 - 5s - 5ms/step - ia: 0.2421 - loss: 0.8748 - mae: 0.7674 - rmse: 0.9088 - smape: 1.6410 - val_ia: 0.1423 - val_loss: 0.8089 - val_mae: 0.7548 - val_rmse: 0.7681 - val_smape: 1.4398

Epoch 3/128                                                                          

922/922 - 5s - 5ms/step - ia: 0.2416 - loss: 0.8669 - mae: 0.7622 - rmse: 0.9053 - smape: 1.6316 - val_ia: 0.1421 - val_loss: 0.8115 - val_mae: 0.7562 - val_rmse: 0.7695 - val_smape: 1.4496

Epoch 4/128                                                                          

922/922 - 5s - 6ms/step - ia: 0.2472 - loss: 0.8554 - mae: 0.7589 - rmse: 0.899

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



58/58 - 4s - 65ms/step - ia: 0.2341 - loss: 0.8773 - mae: 0.7628 - rmse: 0.9343 - smape: 1.5099 - val_ia: 0.3539 - val_loss: 0.9198 - val_mae: 0.8018 - val_rmse: 0.9474 - val_smape: 1.6266

Epoch 2/128                                                                          

58/58 - 1s - 18ms/step - ia: 0.2573 - loss: 0.7670 - mae: 0.7197 - rmse: 0.8748 - smape: 1.4770 - val_ia: 0.3670 - val_loss: 0.7922 - val_mae: 0.7364 - val_rmse: 0.8789 - val_smape: 1.4135

Epoch 3/128                                                                          

58/58 - 1s - 20ms/step - ia: 0.2993 - loss: 0.7103 - mae: 0.6929 - rmse: 0.8412 - smape: 1.4139 - val_ia: 0.3737 - val_loss: 0.7164 - val_mae: 0.7009 - val_rmse: 0.8353 - val_smape: 1.3245

Epoch 4/128                                                                          

58/58 - 1s - 18ms/step - ia: 0.3485 - loss: 0.6534 - mae: 0.6606 - rmse: 0.8069 - smape: 1.3424 - val_ia: 0.3797 - val_loss: 0.6200 - val_mae: 0.6567 - val_rmse: 0.7771 

In [15]:
print(best)

{'activation': 0, 'batch': 0, 'dropout': 0.2, 'layers': 3.0, 'learning_rate': 0.002091021650264381, 'units': 2}


In [16]:
#{'activation': 0, 'batch': 0, 'dropout': 0.2, 'layers': 3.0, 'learning_rate': 0.002091021650264381, 'units': 2}